# MLOps Assignment: Hugging Face Fine-Tuning, Experiment Tracking & Deployment

**Objective:** To build a complete MLOps pipeline by fine-tuning a pre-trained language model, tracking the experiment with Weights & Biases (W&B), and deploying the final model to the Hugging Face Hub.

### Task 1: Environment Setup & Secrets Configuration

**What we are doing:**
Before we begin, we need to ensure our environment is equipped with the latest libraries and authenticated to external services. 

**What we have done so far:**
* Configured the Kaggle environment to use hardware acceleration (`GPU T4 x2`) to speed up model training.
* Enabled the `Internet` toggle to allow downloading pre-trained models and pushing artifacts to external hubs.
* Securely stored our `WANDB_API_KEY` and `HF_TOKEN` using Kaggle Secrets to prevent hardcoding sensitive information in our scripts.

**Steps:**
1. **Install Dependencies:** We will upgrade the `transformers` and `wandb` (Weights & Biases) libraries to ensure we have the latest features for tracking and Hugging Face integration.
2. **Configure Secrets:** We will initialize the `KaggleSecrets` client to securely load our `WANDB_API_KEY` and `HF_TOKEN` as environment variables. This ensures that the `wandb` and `huggingface_hub` libraries can authenticate automatically in the background and also prevents hardcoding sensitive API keys in plain text, which is a critical MLOps security practice.

In [1]:
!pip install -U transformers wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 95.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 54.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transform

In [2]:
# Initialize securely stored API tokens
from kaggle_secrets import UserSecretsClient
import os

print("Fetching secrets...")
secrets = UserSecretsClient()

# Load tokens
WANDB_API_KEY = secrets.get_secret('WANDB_API_KEY')
HF_TOKEN = secrets.get_secret('HF_TOKEN')

# Set as environment variables for library access
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_TOKEN'] = HF_TOKEN

print("Secrets loaded and environment variables set successfully!")

Fetching secrets...
Secrets loaded and environment variables set successfully!


### Dataset Preparation & Preprocessing

**What we are doing:**
Before initializing our model, we must fetch and preprocess our training data. We are using a subset of the UCSD Goodreads Review dataset, focusing on categorizing reviews by genre.

**Steps:**
1. **Data Ingestion:** We download compressed JSON files containing genre-specific reviews directly from the source URLs.
2. **Sampling & Splitting:** To ensure our training loop runs efficiently within Kaggle's GPU limits, we sample a small subset of reviews for each genre and split them into training and testing sets (e.g., 800 for training, 200 for testing per genre).
3. **Label Encoding:** Language models require numerical labels. We extract the unique text genres and create mapping dictionaries (`label2id` and `id2label`). This is critical for configuring the classification head of our model later.

In [3]:
# Basic Python modules
from collections import defaultdict
import random
import pickle

# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# For working with gzip files
# https://docs.python.org/3/library/gzip.html
import gzip

# For working with JSON files
import json

# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# For deep learning
# https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html
import torch

# For plotting and data visualization
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker
sns.set(style='ticks', font_scale=1.2)

In [4]:
# This is where our target data is hosted on the web. You only need these paths for the book review dataset.

# Source: https://mengtingwan.github.io/data/goodreads.html#datasets

genre_url_dict = {'poetry':                 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz',
                  'children':               'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz',
                  'comics_graphic':         'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz',
                  'fantasy_paranormal':     'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz',
                  'history_biography':      'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz',
                  'mystery_thriller_crime': 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz',
                  'romance':                'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz',
                  'young_adult':            'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz'}

In [5]:
import requests
# Stream reviews from URL and collect a subset
def load_reviews(url, head=10000, sample_size=2000):
    reviews = []
    count = 0

    response = requests.get(url, stream=True)
    print(response)
    with gzip.open(response.raw, 'rt', encoding='utf-8') as file:
        for line in file:
            d = json.loads(line)
            reviews.append(d['review_text'])
            count += 1

            # Stop if we have reached the 100,000 limit
            if head is not None and count >= head:
                break

    # Return random sample of reviews
    return random.sample(reviews, min(sample_size, len(reviews)))

# Reviews by genre
genre_reviews_dict = {}

# Load reviews for each genre
for genre, url in genre_url_dict.items():
    print(f'Loading reviews for genre: {genre}')
    genre_reviews_dict[genre] = load_reviews(url, head=10000, sample_size=2000)


Loading reviews for genre: poetry
<Response [200]>
Loading reviews for genre: children
<Response [200]>
Loading reviews for genre: comics_graphic
<Response [200]>
Loading reviews for genre: fantasy_paranormal
<Response [200]>
Loading reviews for genre: history_biography
<Response [200]>
Loading reviews for genre: mystery_thriller_crime
<Response [200]>
Loading reviews for genre: romance
<Response [200]>
Loading reviews for genre: young_adult
<Response [200]>


In [6]:
train_texts = []
train_labels = []

test_texts = []
test_labels = []

for _genre, _reviews in genre_reviews_dict.items():

  _reviews = random.sample(_reviews, 1000) # Use a very small set as an example.

  for _review in _reviews[:800]:
    train_texts.append(_review)
    train_labels.append(_genre)
  for _review in _reviews[800:]:
    test_texts.append(_review)
    test_labels.append(_genre)

In [7]:
unique_labels = set(label for label in train_labels)
label2id = {label: id for id, label in enumerate(unique_labels)}
id2label = {id: label for label, id in label2id.items()}
print("Labels successfully encoded! id2label is ready.")

Labels successfully encoded! id2label is ready.


### Task 2: Load a Pre-trained Model from Hugging Face

**What we are doing:**
We need to select a pre-trained language model, load its tokenizer, and initialize it for sequence classification with the correct number of output labels. 

**Model Selection Rationale (DistilBERT):**
For this project, we have chosen `distilbert-base-cased`. DistilBERT is a distilled version of the BERT model that is 40% smaller and 60% faster, while retaining over 95% of the original model's language understanding capabilities. In a resource-constrained MLOps environment like Kaggle, where GPU hours and memory (OOM errors) are practical bottlenecks, DistilBERT provides an optimal balance between rapid training iteration and high accuracy. It serves as an excellent "text engine" for our pipeline without the computational overhead of full-sized BERT.

**Next Steps:**
In the code cell below, we will import the necessary `transformers` classes, configure our GPU device, and load the `DistilBertTokenizerFast`. We will also prepare the model initialization snippet.

In [8]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# Setup device to use Kaggle's GPU T4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define the model name and max sequence length
model_name = 'distilbert-base-cased'
max_length = 512

# Load the Tokenizer
print(f"Loading tokenizer for {model_name}...")
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
print("Tokenizer loaded successfully!")

# Load the Model 
# (This assumes 'id2label' is already defined in the starter notebook's dataset loading steps.
# If it throws a NameError here, run the starter notebook cells that define the dataset first, then re-run this).
try:
    print("Loading model...")
    model = DistilBertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(id2label) 
    ).to(device)
    print("Model loaded successfully to GPU!")
except NameError:
    print("\nNote: 'id2label' is not defined yet. Please run the dataset preparation cells from your starter notebook first, then come back and run this cell to initialize the model with the correct 'num_labels'.")

Using device: cuda
Loading tokenizer for distilbert-base-cased...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Loading model...


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully to GPU!


### Tokenizing the Dataset

**What we are doing:**
Language models cannot process raw text strings. We must use our loaded `DistilBertTokenizerFast` to convert our `train_texts` and `test_texts` into numerical token IDs and attention masks. We will then wrap these encodings and our numerical labels into a custom PyTorch `Dataset` object, which is the exact format required by the Hugging Face `Trainer`.

In [9]:
import torch

print("Tokenizing training and testing texts...")

train_encodings = tokenizer(
    train_texts, 
    truncation=True, 
    padding=True, 
    max_length=max_length
)
test_encodings = tokenizer(
    test_texts, 
    truncation=True, 
    padding=True, 
    max_length=max_length
)

class GoodreadsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels, label2id):
        self.encodings = encodings
        self.labels = labels
        self.label2id = label2id

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.label2id[self.labels[idx]])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = GoodreadsDataset(train_encodings, train_labels, label2id)
test_dataset = GoodreadsDataset(test_encodings, test_labels, label2id)

print("Datasets successfully tokenized and formatted for PyTorch!")

Tokenizing training and testing texts...
Datasets successfully tokenized and formatted for PyTorch!


### Task 3: Train the Model on Kaggle & Track with W&B

**What we are doing:**
Now that our data is tokenized and our model is loaded with a fresh classification head, we will fine-tune the model using Hugging Face's `Trainer` API. We will integrate Weights & Biases (W&B) to log our training metrics (loss, accuracy, F1-score) and system resource utilization (GPU usage) in real-time.

**Next Steps:**
In the code cell below, we initialize a W&B run and define a `compute_metrics` function to evaluate our model. We then set up our `TrainingArguments` (ensuring `report_to="wandb"` is set) and execute `trainer.train()`. This will kick off the fine-tuning process on Kaggle's GPU.

In [10]:
import wandb
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Initialize Weights & Biases run
wandb.init(
    project="mlops-assignment2",
    name="distilbert-fine-tuning",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "max_length": max_length,
        "dataset": "UCSD Goodreads Genre Subset",
    }
)

# Define evaluation metrics to calculate at the end of each epoch
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

# Configure training parameters and explicitly report to W&B
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="distilbert-fine-tuning"
)

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting model training...")

# Start the actual fine-tuning loop
trainer.train()

print("Training complete!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: zeeshu-irritant (zeeshu-irritant-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting model training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.532681,2.570932,0.546250,0.544522
2,1.929972,2.481054,0.546875,0.555762
3,1.424284,2.497455,0.565000,0.566602


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


### Task 4: Evaluate the Model & Save Results

**What we are doing:**
While W&B logged our high-level accuracy and F1-score during training, a robust MLOps pipeline requires a granular look at how the model performs across *each specific class* (genre). 

**Next Steps:**
We will use the model to predict the genres of our unseen `test_dataset`. We will generate a comprehensive scikit-learn `classification_report`, save it as a structured JSON file locally, and upload it as a tracked "Artifact" to our Weights & Biases project. This ensures that the exact evaluation metrics are permanently version-controlled alongside the model run.

In [11]:
import json
from sklearn.metrics import classification_report

print("Evaluating model on the test dataset...")
eval_results = trainer.predict(test_dataset)

# Extract predictions and true labels
y_pred = np.argmax(eval_results.predictions, axis=-1)
y_true = eval_results.label_ids

# Map numeric IDs back to text genres for a readable report
target_names = [id2label[i] for i in range(len(id2label))]

# Generate a detailed classification report as a dictionary
report_dict = classification_report(
    y_true, 
    y_pred, 
    target_names=target_names, 
    output_dict=True
)

# Save the dictionary as a local JSON file
with open("classification_report.json", "w") as file:
    json.dump(report_dict, file, indent=4)
    
print("Classification report saved locally as 'classification_report.json'.")

# Log the saved JSON file to Weights & Biases as a tracked artifact
print("Uploading evaluation artifact to W&B...")
artifact = wandb.Artifact(name="evaluation_metrics", type="metrics")
artifact.add_file("classification_report.json")
wandb.log_artifact(artifact)

print("Evaluation successfully logged to Weights & Biases!")

Evaluating model on the test dataset...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Classification report saved locally as 'classification_report.json'.
Uploading evaluation artifact to W&B...
Evaluation successfully logged to Weights & Biases!


### Task 5: Push the Model to Hugging Face Hub

**What we are doing:**
The final stage of our MLOps pipeline is deployment. We will upload our fine-tuned model weights and our customized tokenizer to the Hugging Face Hub. 

**Next Steps:**
Using the `HF_TOKEN` we configured in Task 1, we will authenticate with Hugging Face and push our assets to a public repository. This makes our model accessible via the cloud for inference, sharing, or further fine-tuning by other developers.

In [12]:
import os
from huggingface_hub import login

# Authenticate with Hugging Face using the token loaded in Task 1
login(token=os.environ['HF_TOKEN'])

# Define your repository name (CHANGE 'your-hf-username' TO YOUR ACTUAL USERNAME)
hf_username = "zeeshan-hf"
repo_name = f"{hf_username}/distilbert-goodreads-genres"

print(f"Creating repository and pushing to {repo_name}...")

# Push the model and the tokenizer to the Hub
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Deployment complete! View your model at: https://huggingface.co/{repo_name}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Creating repository and pushing to zeeshan-hf/distilbert-goodreads-genres...


README.md:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/2.42k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Deployment complete! View your model at: https://huggingface.co/zeeshan-hf/distilbert-goodreads-genres


### Final W&B Logging & Cleanup

**What we are doing:**
To fulfill the final tracking requirements of our MLOps pipeline, we need to explicitly log our evaluation metrics and our deployed model's URL to Weights & Biases. Finally, we will gracefully close the W&B run.

**Next Steps:**
We will extract the loss, accuracy, and F1-score from our `eval_results` object and log them to W&B. We will also save the Hugging Face Hub link to the W&B run summary for complete traceability, and then call `wandb.finish()`.

In [13]:
import wandb

# Log final metrics to W&B explicitly (Task 3 requirement)
wandb.log({
    "final/loss": eval_results.metrics.get("test_loss", 0),
    "final/accuracy": eval_results.metrics.get("test_accuracy", 0),
    "final/f1": eval_results.metrics.get("test_f1", 0)
})

# Record the Hugging Face link in W&B summary (Task 4 requirement)
wandb.run.summary["huggingface_model"] = f"https://huggingface.co/{repo_name}"

# End W&B run
wandb.finish()

print("Final W&B logging complete and run finished!")

eval/accuracy,▁▁█
eval/f1,▁▅█
eval/loss,█▁▂
eval/runtime,▁▄█
eval/samples_per_second,█▅▁
eval/steps_per_second,█▅▁
final/accuracy,▁
final/f1,▁
final/loss,▁
test/accuracy,▁
+10,...


Final W&B logging complete and run finished!


### Conclusion

This concludes the end-to-end MLOps pipeline for Goodreads Genre Classification. 

**Summary of Achievements:**
1. **Environment & Security:** Successfully initialized Kaggle GPU resources and securely managed API tokens using Kaggle Secrets.
2. **Data & Modeling:** Downsampled and tokenized the dataset, and initialized a lightweight `distilbert-base-cased` model with a custom classification head.
3. **Training & Tracking:** Fine-tuned the model while streaming real-time telemetry (loss, GPU utilization, etc.) to Weights & Biases.
4. **Evaluation:** Generated a granular classification report and versioned it as a W&B Artifact.
5. **Deployment:** Pushed the final model weights and tokenizer to the Hugging Face Hub, making the model accessible for downstream production applications.

The pipeline is fully reproducible, tracked, and deployed!